# Week 43 extension: calibrated abstention

**Hypothesis (stated before running).** The integrated QA model uses one global no-answer threshold, but the
validation languages differ strongly in how often questions are unanswerable (Arabic 12.5%, Korean 5.3%,
Telugu 24.2%) and the training split differs again (Telugu 3.3%). We expect the no-answer score to be shifted
differently per language, so **language-specific thresholds should beat a single global threshold**.

- Primary metric: overall answerability macro-F1 under cross-validation within the validation set.
- Expected direction: per-language > global, by at least 0.01 macro-F1 overall, with the gain concentrated in
  Telugu (higher unanswerable recall) and answerable recall falling by less than 0.03.
- **Falsified if** the overall gain is below 0.01, or Telugu does not improve, or per-language is not better in
  most repeats.
- Secondary question: does a language-specific intercept in Platt scaling reduce ECE / Brier score?

Protocol: 5-fold stratified (language x label) cross-validation inside validation, repeated 5 times with fixed
seeds. Each threshold is fitted on four folds and scored on the held-out fold; out-of-fold predictions are pooled
per repeat. The challenge set is not used.

In [ ]:
import nb_setup
import pandas as pd
from src.calibration import (
    cross_validate, summarise, paired_difference, evaluate_at_threshold, best_threshold,
)

df = pd.read_csv("../data/qa_val_predictions.csv", keep_default_na=False)
df["gold_answerable"] = df["gold_answerable"].astype(int)
print(len(df), "examples;", df.groupby("lang")["gold_answerable"].agg(["size", "mean"]).round(3).to_dict("index"))

## Reference: the notebook-7 threshold, fitted and scored on all of validation (optimistic)

In [ ]:
# -5.895 maximised token F1 on all of validation in notebook 7
evaluate_at_threshold(df, -5.895390)

## Cross-validated comparison

In [ ]:
results = cross_validate(df, n_splits=5, n_repeats=5, seed=42)
decision = results[results["method"].isin(["global", "per_language"])]
summarise(decision, ["macro_f1", "unans_precision", "unans_recall", "ans_recall", "squad_em", "squad_f1"])

In [ ]:
for metric in ["macro_f1", "unans_recall", "ans_recall", "squad_f1"]:
    print(metric, "(per_language - global)")
    display(paired_difference(decision, "per_language", "global", metric))

## Calibration quality (lower is better)

In [ ]:
calibration = results[results["method"].str.startswith("platt")]
summarise(calibration, ["ece", "brier"])

In [ ]:
paired_difference(calibration, "platt_per_language", "platt_global", "ece")

## Interpretation

Fill in after running, against the hypothesis above: did the overall and Telugu gains reach the stated size,
what happened to answerable recall, and in how many of the 5 repeats was per-language better? Also report the
fitted per-language thresholds on all of validation:

In [ ]:
from src.calibration import PerLanguageThreshold, GlobalThreshold
y, s, l = df["gold_answerable"].values, df["no_answer_score"].values, df["lang"].values
print("global:", round(GlobalThreshold().fit(s, y, l).threshold, 3))
print("per language:", {k: round(v, 3) for k, v in PerLanguageThreshold().fit(s, y, l).thresholds.items()})